# Feature Engineering

Feature engineering turns raw columns into representations that make patterns easier for a model to
learn. Good features often matter more than the choice of algorithm.

This notebook demonstrates:

- date and time features,
- ratio and interaction features,
- binning,
- categorical encoding, including target encoding,
- polynomial features,
- comparing a baseline pipeline with an engineered pipeline.

## Learning objectives

1. Create features from dates, ratios, and interactions.
2. Encode categorical variables appropriately.
3. Apply target encoding without leakage.
4. Compare models with and without engineered features.
5. Recognise feature-engineering pitfalls.

## 1. Concepts

### Why engineer features

A model can only learn from the representation it receives. A raw timestamp does not directly express
"weekend" or "days since signup"; a ratio can express affordability better than two separate columns.

### Common transformations

- **Date/time**: year, month, weekday, is_weekend, days since an event, time since last activity.
- **Ratios**: income per product, balance / salary, events per day.
- **Interactions**: product of two features, differences, sums.
- **Binning**: age groups, tenure bands; useful for non-linear effects and interpretability.
- **Encoding**: one-hot for low-cardinality categories, ordinal for ordered categories, target
  encoding for high-cardinality categories.
- **Polynomial features**: squares and cross-products for linear models.

### Leakage

Any transformation that uses the target must be fitted inside the training folds. Target encoding is
the classic example: computing category means on the full dataset leaks the target into validation.

### Trees vs linear models

Trees are invariant to monotonic transformations and do not need scaling. Linear models benefit from
scaling, interactions, and non-linear transformations.

## 2. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

## 3. Build a synthetic customer-churn dataset

In [2]:
rng = np.random.default_rng(RANDOM_STATE)
n = 3000

age = rng.integers(18, 70, n)
income = rng.normal(55000, 18000, n).clip(15000, 160000)
tenure = rng.integers(1, 72, n)
products = rng.integers(1, 6, n)
city = rng.choice(["Mumbai", "Delhi", "Bengaluru", "Pune", "Chennai", "Hyderabad", "Kolkata"], n)
signup_date = pd.to_datetime("2022-01-01") + pd.to_timedelta(rng.integers(0, 900, n), unit="D")
is_active = rng.random(n) < 0.65

logit = (
    -1.2
    - 0.035 * (tenure - 24)
    + 0.35 * (products - 3)
    - 0.00001 * (income - 55000)
    + 0.9 * (~is_active)
    + 0.25 * (city == "Mumbai")
    + rng.normal(0, 0.6, n)
)
churn_probability = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, churn_probability)

df = pd.DataFrame({
    "age": age,
    "income": income.round(2),
    "tenure_months": tenure,
    "products": products,
    "city": city,
    "signup_date": signup_date,
    "is_active": is_active.astype(int),
    "churn": churn,
})

df.head()

,age,income,tenure_months,products,city,signup_date,is_active,churn
0,22,72333.29,41,5,Kolkata,2022-11-29,1,1
1,58,41957.00,47,5,Kolkata,2023-06-27,1,0
2,52,39632.33,70,2,Kolkata,2022-09-01,0,0
3,40,63274.71,54,3,Mumbai,2022-11-19,1,0
4,40,72793.96,16,4,Chennai,2022-10-25,0,1


In [3]:
print(df.shape)
print(df["churn"].value_counts(normalize=True).round(3))
df.describe(include="all").T.head(12)

(3000, 8)
churn
0    0.726
1    0.274
Name: proportion, dtype: float64


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
age,3000.0,NaN,NaN,NaN,43.549667,18.0,30.0,43.0,57.0,69.0,15.054714
income,3000.0,NaN,NaN,NaN,54749.718437,15000.0,42502.345,54845.075,66429.8875,117172.84,17825.763212
tenure_months,3000.0,NaN,NaN,NaN,36.132,1.0,19.0,36.0,53.25,71.0,20.200064
products,3000.0,NaN,NaN,NaN,2.952667,1.0,2.0,3.0,4.0,5.0,1.411532
city,3000,7,Bengaluru,458,NaN,NaN,NaN,NaN,NaN,NaN,NaN
signup_date,3000,NaN,NaN,NaN,2023-03-18 23:52:48.000000256,2022-01-01 00:00:00,2022-08-02 00:00:00,2023-03-19 00:00:00,2023-10-27 06:00:00,2024-06-18 00:00:00,NaN
is_active,3000.0,NaN,NaN,NaN,0.656333,0.0,0.0,1.0,1.0,1.0,0.475011
churn,3000.0,NaN,NaN,NaN,0.274,0.0,0.0,0.0,1.0,1.0,0.446083


## 4. Baseline: raw numeric columns + one-hot city

In [4]:
X = df.drop(columns="churn")
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

baseline_preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), ["age", "income", "tenure_months", "products", "is_active"]),
    ("city", OneHotEncoder(handle_unknown="ignore"), ["city"]),
])

baseline = Pipeline([
    ("preprocess", baseline_preprocess),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

baseline_scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
print(f"Baseline CV ROC-AUC: {baseline_scores.mean():.4f} (+/- {baseline_scores.std():.4f})")

Baseline CV ROC-AUC: 0.7326 (+/- 0.0178)


## 5. Engineer date, ratio, and interaction features

In [5]:
def engineer_features(frame):
    engineered = frame.copy()
    engineered["signup_year"] = engineered["signup_date"].dt.year
    engineered["signup_month"] = engineered["signup_date"].dt.month
    engineered["signup_weekday"] = engineered["signup_date"].dt.weekday
    engineered["is_weekend"] = (engineered["signup_weekday"] >= 5).astype(int)
    engineered["days_since_signup"] = (
        pd.Timestamp("2025-01-01") - engineered["signup_date"]
    ).dt.days

    engineered["income_per_product"] = engineered["income"] / engineered["products"]
    engineered["tenure_per_product"] = engineered["tenure_months"] / engineered["products"]
    engineered["monthly_income"] = engineered["income"] / 12
    engineered["engagement_score"] = engineered["is_active"] * engineered["tenure_months"]

    engineered["age_group"] = pd.cut(
        engineered["age"], bins=[17, 25, 35, 45, 55, 70],
        labels=["18-25", "26-35", "36-45", "46-55", "56-70"],
    )
    engineered["tenure_band"] = pd.cut(
        engineered["tenure_months"], bins=[0, 12, 24, 48, 72],
        labels=["0-12", "13-24", "25-48", "49-72"],
    )
    return engineered

df_engineered = engineer_features(df)
df_engineered.head()

,age,income,tenure_months,products,city,signup_date,is_active,churn,signup_year,signup_month,signup_weekday,is_weekend,days_since_signup,income_per_product,tenure_per_product,monthly_income,engagement_score,age_group,tenure_band
0,22,72333.29,41,5,Kolkata,2022-11-29,1,1,2022,11,1,0,764,14466.658,8.2,6027.774167,41,18-25,25-48
1,58,41957.00,47,5,Kolkata,2023-06-27,1,0,2023,6,1,0,554,8391.400,9.4,3496.416667,47,56-70,25-48
2,52,39632.33,70,2,Kolkata,2022-09-01,0,0,2022,9,3,0,853,19816.165,35.0,3302.694167,0,46-55,49-72
3,40,63274.71,54,3,Mumbai,2022-11-19,1,0,2022,11,5,1,774,21091.570,18.0,5272.892500,54,36-45,49-72
4,40,72793.96,16,4,Chennai,2022-10-25,0,1,2022,10,1,0,799,18198.490,4.0,6066.163333,0,36-45,13-24


## 6. Target encoding without leakage

`TargetEncoder` from scikit-learn uses cross-fitting during `fit_transform`, which reduces leakage.
It is useful for high-cardinality categories such as city, postcode, or customer segment.

We compare one-hot and target encoding for `city`.

In [6]:
numeric_features = [
    "age", "income", "tenure_months", "products", "is_active",
    "signup_year", "signup_month", "signup_weekday", "is_weekend", "days_since_signup",
    "income_per_product", "tenure_per_product", "monthly_income", "engagement_score",
]
categorical_features = ["age_group", "tenure_band"]
target_encoded_features = ["city"]

X_eng = df_engineered.drop(columns=["churn", "signup_date"])
y_eng = df_engineered["churn"]

X_eng_train, X_eng_test, y_eng_train, y_eng_test = train_test_split(
    X_eng, y_eng, test_size=0.2, random_state=RANDOM_STATE, stratify=y_eng
)

engineered_preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("city_target", TargetEncoder(target_type="binary", random_state=RANDOM_STATE), target_encoded_features),
])

engineered = Pipeline([
    ("preprocess", engineered_preprocess),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

engineered_scores = cross_val_score(
    engineered, X_eng_train, y_eng_train, cv=5, scoring="roc_auc", n_jobs=-1
)
print(f"Engineered CV ROC-AUC: {engineered_scores.mean():.4f} (+/- {engineered_scores.std():.4f})")

Engineered CV ROC-AUC: 0.7267 (+/- 0.0163)


## 7. Polynomial features for linear models

Polynomial features add squares and interactions. They can capture non-linearity, but the number of
features grows quickly and regularisation becomes important.

In [7]:
from sklearn.preprocessing import PolynomialFeatures

poly_numeric = Pipeline([
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
])

poly_preprocess = ColumnTransformer([
    ("numeric", poly_numeric, ["age", "income", "tenure_months", "products", "is_active"]),
    ("city", OneHotEncoder(handle_unknown="ignore"), ["city"]),
])

poly_model = Pipeline([
    ("preprocess", poly_preprocess),
    ("model", LogisticRegression(max_iter=5000, C=0.1, random_state=RANDOM_STATE)),
])

poly_scores = cross_val_score(poly_model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
print(f"Polynomial CV ROC-AUC: {poly_scores.mean():.4f} (+/- {poly_scores.std():.4f})")

Polynomial CV ROC-AUC: 0.7355 (+/- 0.0187)


## 8. Compare models on the held-out test set

In [8]:
comparison = {}

baseline.fit(X_train, y_train)
comparison["Baseline (raw)"] = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])

engineered.fit(X_eng_train, y_eng_train)
comparison["Engineered"] = roc_auc_score(y_eng_test, engineered.predict_proba(X_eng_test)[:, 1])

poly_model.fit(X_train, y_train)
comparison["Polynomial"] = roc_auc_score(y_test, poly_model.predict_proba(X_test)[:, 1])

forest = Pipeline([
    ("preprocess", ColumnTransformer([
        ("numeric", "passthrough", ["age", "income", "tenure_months", "products", "is_active"]),
        ("city", OneHotEncoder(handle_unknown="ignore"), ["city"]),
    ])),
    ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
])
forest.fit(X_train, y_train)
comparison["Random forest (raw)"] = roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1])

pd.Series(comparison).sort_values(ascending=False).round(4)

Baseline (raw)         0.7221
Engineered             0.7196
Polynomial             0.7122
Random forest (raw)    0.6882
dtype: float64

## 9. Feature-engineering checklist

- Understand the column before transforming it.
- Keep transformations inside a pipeline so cross-validation is honest.
- Avoid target leakage: no future information, no target-derived features outside training folds.
- Prefer simple, explainable features when predictive performance is similar.
- Watch the feature count; more features increase overfitting and maintenance cost.
- Validate on a time-based split if the data is temporal.
- Document every feature so others can reproduce it.

## Key takeaways

- Dates, ratios, interactions, and bins encode domain knowledge.
- One-hot encoding is a good default for low-cardinality categories.
- Target encoding is powerful but must be cross-fitted.
- Polynomial features help linear models but need regularisation.
- More features are not automatically better; validate with cross-validation.

## Exercises

1. Add a feature for the number of days since the last activity and evaluate it.
2. Replace target encoding with frequency encoding and compare.
3. Use `OrdinalEncoder` for `tenure_band` and compare with one-hot.
4. Add a `salary_band` feature and test whether it improves the random forest.
5. Introduce a deliberately leaky feature and observe the inflated cross-validation score.